# Condor Pipeline — Unified Analysis

This notebook loads the aggregated results from the Condor pipeline and
reproduces **all plots** from both `characterisation.ipynb` and
`hit_competition_study.ipynb`.

## Workflow
1. Generate params & submit: `./submit.sh`
2. Wait for all jobs: `condor_q`
3. Aggregate: `python scripts/aggregate.py --results-dir results`
4. Open this notebook and run all cells.

All plots work entirely from saved aggregated data — no event generation required.

In [ ]:
# ── Imports & setup ──────────────────────────────────────────────
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.lines import Line2D

%matplotlib inline
plt.rcParams["figure.dpi"] = 120
plt.rcParams.update({
    'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 14,
    'xtick.labelsize': 12, 'ytick.labelsize': 12, 'legend.fontsize': 11,
    'axes.linewidth': 1.2, 'lines.linewidth': 2, 'lines.markersize': 8,
})

# ── Paths ──
AGG_DIR = Path("results/aggregated")
assert AGG_DIR.exists(), f"Aggregated results not found at {AGG_DIR.resolve()}.\nRun aggregate.py first!"

FIGURES_DIR = Path("results/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name):
    """Save figure as PDF and PNG."""
    fig.savefig(FIGURES_DIR / f"{name}.pdf", bbox_inches='tight', dpi=150)
    fig.savefig(FIGURES_DIR / f"{name}.png", bbox_inches='tight', dpi=150)

def load_json(path):
    with open(path) as f:
        return json.load(f)

print(f"Aggregated data: {AGG_DIR.resolve()}")
print(f"Figures output:  {FIGURES_DIR.resolve()}")

In [ ]:
# ── Load run status & parameters ─────────────────────────────
status = load_json(AGG_DIR / "run_status.json")
run_summary = load_json(AGG_DIR / "run_summary.json")
params = run_summary.get("parameters", {})

print(f"Jobs: {status['completed']} completed, {status['failed']} failed, {status['missing']} pending")
print(f"Task breakdown:")
for task, count in status.get('tasks_found', {}).items():
    print(f"  {task:20s}: {count}")

# ── Physics constants (same as helpers.py) ──
SIGMA_RES = 0.0
SIGMA_SCATT = 1e-4
DZ_MM = 33.0
SCALE = 3.0
GAMMA = 1.5
DELTA = 1.0
BASELINE = DELTA / (DELTA + GAMMA)
THRESHOLD = (1 + BASELINE) / 2

def compute_epsilon(sigma_res, sigma_scatt, dz, scale=1.0, theta_min=1.5e-5):
    theta_s = scale * sigma_scatt
    theta_r = np.arctan((scale * sigma_res) / dz) if dz != 0 else 0.0
    return float(np.sqrt(2 * theta_s**2 + 12 * theta_r**2 + 2 * theta_min**2))

EPSILON = compute_epsilon(SIGMA_RES, SIGMA_SCATT, DZ_MM, scale=SCALE)

# Standard parameter values
ANGLE_SETTINGS = params.get('angle_settings', [0.2, 0.1, 0.04])
SCATT_MULTIPLIERS = params.get('scatt_multipliers', [1, 2, 4])
TRACK_SIZES = params.get('track_sizes', list(range(10, 110, 10)))
TRACK_DENSITIES = params.get('track_densities', [5, 10, 20, 30, 50, 75, 100, 150])
ROC_DENSITIES = params.get('roc_densities', [10, 30, 50, 100, 150])
N_REPEATS_SCAN = params.get('n_repeats_scan', 5)

# Epsilon per scattering multiplier
all_epsilons = {mult: compute_epsilon(SIGMA_RES, SIGMA_SCATT * mult, DZ_MM, scale=SCALE)
                for mult in SCATT_MULTIPLIERS}

# Angle labels
ANGLE_LABELS = {a: f'±{a}' for a in ANGLE_SETTINGS}

print(f"\nEpsilon (1x): {EPSILON*1e3:.3f} mrad")
print(f"Threshold:    {THRESHOLD:.3f}")

---
# Part 1: Characterisation Plots

Reproduces all plots from `characterisation.ipynb`:
- 100×50 bulk pairwise segment angle distributions
- Parametric scan: segment metrics vs event size (3 angle × 3 scattering)
- Acceptance histograms: true vs false angles per (angle, scattering) setting

In [ ]:
# ── Load 100×50 bulk angle distributions ─────────────────────
bulk_npz = np.load(str(AGG_DIR / "char_bulk_angles.npz"))
all_true_angles = bulk_npz["all_true_angles"]
all_false_angles = bulk_npz["all_false_angles"]
print(f"Bulk angles: {len(all_true_angles):,} true, {len(all_false_angles):,} false")

In [ ]:
# ── Plot: Pairwise segment angle histograms (100×50) ─────────
bins = np.linspace(0, 0.5, 120)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

axes[0].hist(all_true_angles, bins=bins, color='forestgreen', alpha=0.8,
             edgecolor='black', lw=0.3, label=f'True ({len(all_true_angles):,})')
axes[0].axvline(EPSILON, color='blue', ls='--', lw=2,
                label=f'epsilon = {EPSILON:.4f}')
axes[0].set_title('True Segment Pairs (same track)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Pairwise angle between segments (rad)')
axes[0].set_ylabel('Count')
axes[0].legend(fontsize=10); axes[0].grid(alpha=0.3)

axes[1].hist(all_false_angles, bins=bins, color='indianred', alpha=0.8,
             edgecolor='black', lw=0.3, label=f'False ({len(all_false_angles):,})')
axes[1].axvline(EPSILON, color='blue', ls='--', lw=2,
                label=f'epsilon = {EPSILON:.4f}')
axes[1].set_title('False Segment Pairs', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Pairwise angle between segments (rad)')
axes[1].set_ylabel('Count')
axes[1].legend(fontsize=10); axes[1].grid(alpha=0.3)

axes[2].hist(all_true_angles, bins=bins, color='forestgreen', alpha=0.6,
             edgecolor='black', lw=0.3, density=True,
             label=f'True ({len(all_true_angles):,})')
axes[2].hist(all_false_angles, bins=bins, color='indianred', alpha=0.5,
             edgecolor='black', lw=0.3, density=True,
             label=f'False ({len(all_false_angles):,})')
axes[2].axvline(EPSILON, color='blue', ls='--', lw=2,
                label=f'epsilon = {EPSILON:.4f}')
axes[2].set_title('True vs False (normalised)', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Pairwise angle between segments (rad)')
axes[2].set_ylabel('Density')
axes[2].legend(fontsize=10); axes[2].grid(alpha=0.3)

N_EVENTS = params.get('n_events_bulk', 100)
N_TRACKS_STUDY = params.get('n_tracks_bulk', 50)
plt.suptitle(f'Pairwise Segment Angles — {N_EVENTS} events × {N_TRACKS_STUDY} tracks',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, 'char_bulk_angles')
plt.show()

In [ ]:
# ── Load parametric scan data ────────────────────────────────
char_scan_raw = load_json(AGG_DIR / "char_scan.json")
char_scan_data = char_scan_raw["scan_data"]   # {angle_str: {mult_str: results_list}}
scan_epsilons = char_scan_raw["epsilons"]

print("Loaded parametric scan:")
for a_str, mult_dict in char_scan_data.items():
    for m_str, results in mult_dict.items():
        eps = scan_epsilons.get(m_str, 0)
        print(f"  angle=±{a_str}, {m_str}x scattering: {len(results)} track sizes, eps={float(eps)*1e3:.3f} mrad")

In [ ]:
# ── Plot: Parametric scan — segment metrics vs event size ────
scatter_colors = {1: '#1b7837', 2: '#2166ac', 4: '#c51b7d'}
scatter_fmts   = {1: 'o-',     2: 's--',     4: 'D:'}

for angle in ANGLE_SETTINGS:
    fig, axes = plt.subplots(2, 2, figsize=(15, 11))

    for mult in SCATT_MULTIPLIERS:
        results = char_scan_data[str(angle)][str(mult)]
        eps = float(scan_epsilons[str(mult)])
        col = scatter_colors[mult]
        fmt = scatter_fmts[mult]
        lbl = f'{mult}x ($\\varepsilon$={eps*1e3:.2f} mrad)'

        x_tracks = np.array([r['n_tracks'] for r in results])

        # (a) Segment Efficiency
        ax = axes[0, 0]
        eff_mean = np.array([r['eff_mean'] for r in results])
        eff_se   = np.array([r['eff_se']   for r in results])
        ax.errorbar(x_tracks, eff_mean, yerr=eff_se, fmt=fmt, color=col,
                    capsize=4, capthick=1.2, markeredgecolor='black',
                    markeredgewidth=0.6, label=lbl)

        # (b) False Rate
        ax = axes[0, 1]
        fr_mean = np.array([r['fr_mean'] for r in results])
        fr_se   = np.array([r['fr_se']   for r in results])
        ax.errorbar(x_tracks, fr_mean, yerr=fr_se, fmt=fmt, color=col,
                    capsize=4, capthick=1.2, markeredgecolor='black',
                    markeredgewidth=0.6, label=lbl)

        # (c) Segment Pair Counts
        ax = axes[1, 0]
        n_true_m  = np.array([r['n_true_mean']  for r in results])
        n_true_e  = np.array([r['n_true_se']    for r in results])
        n_false_m = np.array([r['n_false_mean'] for r in results])
        n_false_e = np.array([r['n_false_se']   for r in results])
        ax.errorbar(x_tracks, n_true_m,  yerr=n_true_e,  fmt=fmt, color=col,
                    capsize=3, capthick=1, markeredgecolor='black',
                    markeredgewidth=0.5, label=f'{mult}x true')
        ax.errorbar(x_tracks, n_false_m, yerr=n_false_e, fmt=fmt, color=col,
                    capsize=3, capthick=1, markeredgecolor='black',
                    markeredgewidth=0.5, alpha=0.4, label=f'{mult}x false')

        # (d) Accepted Pair Counts
        ax = axes[1, 1]
        ta_m = np.array([r['true_acc_mean']  for r in results])
        ta_e = np.array([r['true_acc_se']    for r in results])
        fa_m = np.array([r['false_acc_mean'] for r in results])
        fa_e = np.array([r['false_acc_se']   for r in results])
        fa_m_plot = np.where(fa_m > 0, fa_m, 0.5)
        fa_e_plot = np.where(fa_m > 0, fa_e, 0)
        ax.errorbar(x_tracks, ta_m, yerr=ta_e, fmt=fmt, color=col,
                    capsize=3, capthick=1, markeredgecolor='black',
                    markeredgewidth=0.5, label=f'{mult}x true acc')
        ax.errorbar(x_tracks, fa_m_plot, yerr=fa_e_plot, fmt=fmt, color=col,
                    capsize=3, capthick=1, markeredgecolor='black',
                    markeredgewidth=0.5, alpha=0.4, label=f'{mult}x false acc')

    # Axis formatting
    axes[0, 0].axhline(100, color='gray', ls='--', lw=1, alpha=0.5)
    axes[0, 0].set_xlabel('Tracks per event')
    axes[0, 0].set_ylabel('Segment Efficiency (%)')
    axes[0, 0].set_title('(a) Segment Efficiency\n'
                          r'($N_{\mathrm{true\,accepted}} / N_{\mathrm{true\,pairs}}$)',
                          fontweight='bold')
    axes[0, 0].legend(fontsize=10, loc='lower left')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].minorticks_on()
    axes[0, 0].tick_params(which='both', direction='in', top=True, right=True)

    axes[0, 1].set_xlabel('Tracks per event')
    axes[0, 1].set_ylabel('Segment False Rate (%)')
    axes[0, 1].set_title('(b) Segment False Rate\n'
                          r'($N_{\mathrm{false\,accepted}} / N_{\mathrm{all\,accepted}}$)',
                          fontweight='bold')
    axes[0, 1].legend(fontsize=10)
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].minorticks_on()
    axes[0, 1].tick_params(which='both', direction='in', top=True, right=True)

    for ax_idx in [(1, 0), (1, 1)]:
        axes[ax_idx].set_yscale('log')
        axes[ax_idx].set_xlabel('Tracks per event')
        axes[ax_idx].grid(True, alpha=0.3, which='both')
        axes[ax_idx].minorticks_on()
        axes[ax_idx].tick_params(which='both', direction='in', top=True, right=True)

    axes[1, 0].set_ylabel('Number of segment pairs')
    axes[1, 0].set_title('(c) Segment Pair Counts', fontweight='bold')
    axes[1, 0].legend(fontsize=9, ncol=2, loc='upper left')

    axes[1, 1].set_ylabel('Number of pairs accepted')
    axes[1, 1].set_title(r'(d) Accepted Segment Pairs ($\theta \leq \varepsilon$)',
                          fontweight='bold')
    axes[1, 1].legend(fontsize=9, ncol=2, loc='upper left')

    fig.suptitle(f'Segment-Level Metrics vs Event Size — Scattering Comparison\n'
                 f'Generation angle: $\\phi_{{\\max}}=\\theta_{{\\max}}=\\pm${angle} rad  '
                 f'({N_REPEATS_SCAN} repeats per size)',
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    save_fig(fig, f'char_scan_angle_{angle}')
    plt.show()

# Summary tables
for angle in ANGLE_SETTINGS:
    print(f"\n{'*'*100}")
    print(f"  Generation angle: phi_max = theta_max = ±{angle} rad")
    print(f"{'*'*100}")
    for mult in SCATT_MULTIPLIERS:
        eps = float(scan_epsilons[str(mult)])
        results = char_scan_data[str(angle)][str(mult)]
        print(f"\n{'='*95}")
        print(f"  {mult}x scattering  (sigma_scatt={SIGMA_SCATT*mult:.1e}, "
              f"epsilon={eps*1e3:.3f} mrad)")
        print(f"{'='*95}")
        print(f"{'Tracks':>7} {'Eff (%)':>12} {'FR (%)':>12} "
              f"{'True pairs':>14} {'False pairs':>14} "
              f"{'True acc':>14} {'False acc':>14}")
        print('-' * 95)
        for r in results:
            print(f"{r['n_tracks']:7d} "
                  f"{r['eff_mean']:7.1f}+/-{r['eff_se']:.2f} "
                  f"{r['fr_mean']:7.2f}+/-{r['fr_se']:.2f} "
                  f"{r['n_true_mean']:8.0f}+/-{r['n_true_se']:.0f} "
                  f"{r['n_false_mean']:8.0f}+/-{r['n_false_se']:.0f} "
                  f"{r['true_acc_mean']:8.0f}+/-{r['true_acc_se']:.0f} "
                  f"{r['false_acc_mean']:8.0f}+/-{r['false_acc_se']:.0f}")

In [ ]:
# ── Load & plot acceptance histograms ────────────────────────
hist_npz = np.load(str(AGG_DIR / "char_hist.npz"))

x_max_rad = 0.03
bins = np.linspace(0, x_max_rad, 100)

for angle in ANGLE_SETTINGS:
    fig, axes = plt.subplots(len(SCATT_MULTIPLIERS), 3,
                             figsize=(20, 5 * len(SCATT_MULTIPLIERS)))

    for row, mult in enumerate(SCATT_MULTIPLIERS):
        eps = all_epsilons[mult]
        key = f"a{angle}_m{mult}"
        ta = hist_npz[f"{key}_true"]
        fa = hist_npz[f"{key}_false"]

        # (left) True
        ax = axes[row, 0]
        ax.hist(ta, bins=bins, color='forestgreen', alpha=0.8,
                edgecolor='black', lw=0.3, label=f'True ({len(ta):,})')
        ax.axvline(eps, color='blue', ls='--', lw=2,
                   label=f'$\\varepsilon$ = {eps*1e3:.2f} mrad')
        ax.set_title(f'{mult}x scattering - True Segment Pairs',
                     fontsize=13, fontweight='bold')
        ax.set_xlabel('Pairwise angle (rad)')
        ax.set_ylabel('Count')
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

        # (middle) False
        ax = axes[row, 1]
        ax.hist(fa, bins=bins, color='indianred', alpha=0.8,
                edgecolor='black', lw=0.3, label=f'False ({len(fa):,})')
        ax.axvline(eps, color='blue', ls='--', lw=2,
                   label=f'$\\varepsilon$ = {eps*1e3:.2f} mrad')
        ax.set_title(f'{mult}x scattering - False Segment Pairs',
                     fontsize=13, fontweight='bold')
        ax.set_xlabel('Pairwise angle (rad)')
        ax.set_ylabel('Count')
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

        # (right) Overlay normalised
        ax = axes[row, 2]
        ax.hist(ta, bins=bins, color='forestgreen', alpha=0.6,
                edgecolor='black', lw=0.3, density=True,
                label=f'True ({len(ta):,})')
        ax.hist(fa, bins=bins, color='indianred', alpha=0.5,
                edgecolor='black', lw=0.3, density=True,
                label=f'False ({len(fa):,})')
        ax.axvline(eps, color='blue', ls='--', lw=2,
                   label=f'$\\varepsilon$ = {eps*1e3:.2f} mrad')
        ax.set_title(f'{mult}x scattering - Normalised overlay',
                     fontsize=13, fontweight='bold')
        ax.set_xlabel('Pairwise angle (rad)')
        ax.set_ylabel('Density')
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    N_TRACKS_HIST = params.get('n_tracks_hist', 50)
    N_EVENTS_HIST = params.get('n_events_hist', 20)
    fig.suptitle(f'Segment-Pair Angle Distributions — '
                 f'$\\phi_{{\\max}}=\\theta_{{\\max}}=\\pm${angle} rad\n'
                 f'{N_EVENTS_HIST} events × {N_TRACKS_HIST} tracks per scattering setting',
                 fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    save_fig(fig, f'char_hist_angle_{angle}')
    plt.show()

---
# Part 2: Hit-Competition Plots

Reproduces all plots from `hit_competition_study.ipynb`:
- Step 1: Hit occupancy distributions
- Step 2: Activation spectra (true vs false segments)
- Step 3: Per-hit competition analysis
- Step 4: Track-level reconstruction quality
- Step 5: Threshold sensitivity / ROC curves
- Step 6: Scattering comparison (segment metrics + histograms)

In [ ]:
# ── Load hit-competition density data (Steps 1-4) ────────────
hc_density = load_json(AGG_DIR / "hc_density.json")
occ_summary = hc_density["occupancy_summary"]
reco_metrics = hc_density["reco_metrics"]

step1_npz = np.load(str(AGG_DIR / "hc_density_step1.npz"))
step2_npz = np.load(str(AGG_DIR / "hc_density_step2.npz"))
step3_npz = np.load(str(AGG_DIR / "hc_density_step3.npz"))

print(f"Loaded {len(reco_metrics)} reco metric entries")
print(f"Occupancy groups: {len(occ_summary)} angles × {len(list(occ_summary.values())[0])} densities")

In [ ]:
# ── Step 1: Hit Occupancy Distributions ──────────────────────
for angle in ANGLE_SETTINGS:
    a_str = str(angle)
    fig, axes = plt.subplots(2, 4, figsize=(20, 8))
    axes_flat = axes.flatten()

    for i, n_trk in enumerate(TRACK_DENSITIES):
        key = f"a{angle}_n{n_trk}_occ"
        if key not in step1_npz:
            continue
        occ = step1_npz[key]
        ax = axes_flat[i]
        ax.hist(occ, bins=np.arange(0.5, max(occ) + 1.5, 1),
                color='steelblue', edgecolor='black', lw=0.5, alpha=0.8)
        ax.set_title(f'{n_trk} tracks', fontweight='bold')
        ax.set_xlabel('Segments per hit')
        ax.set_ylabel('Count')
        ax.grid(True, alpha=0.3)

        info = occ_summary.get(a_str, {}).get(str(n_trk), {})
        ax.text(0.95, 0.95, f"mean={info.get('mean',0):.1f}\nmax={info.get('max',0)}",
                transform=ax.transAxes, ha='right', va='top', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    fig.suptitle(f'Step 1: Hit Occupancy — angle ±{angle} rad',
                 fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    save_fig(fig, f'hc_step1_occupancy_angle_{angle}')
    plt.show()

In [ ]:
# ── Step 2: Activation Spectra ───────────────────────────────
for angle in ANGLE_SETTINGS:
    fig, axes = plt.subplots(2, 4, figsize=(20, 8))
    axes_flat = axes.flatten()

    for i, n_trk in enumerate(TRACK_DENSITIES):
        key = f"a{angle}_n{n_trk}"
        true_key = f"{key}_true_x"
        false_key = f"{key}_false_x"
        if true_key not in step2_npz:
            continue
        true_x = step2_npz[true_key]
        false_x = step2_npz[false_key]

        ax = axes_flat[i]
        bins_act = np.linspace(0, 1, 50)
        ax.hist(true_x, bins=bins_act, alpha=0.6, color='green',
                edgecolor='black', lw=0.3, label=f'True ({len(true_x):,})')
        ax.hist(false_x, bins=bins_act, alpha=0.5, color='red',
                edgecolor='black', lw=0.3, label=f'False ({len(false_x):,})')
        ax.axvline(THRESHOLD, color='blue', ls='--', lw=1.5,
                   label=f'thr={THRESHOLD:.2f}')
        ax.set_title(f'{n_trk} tracks', fontweight='bold')
        ax.set_xlabel('Activation x')
        ax.set_ylabel('Count')
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

    fig.suptitle(f'Step 2: Activation Spectrum — angle ±{angle} rad',
                 fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    save_fig(fig, f'hc_step2_activation_angle_{angle}')
    plt.show()

In [ ]:
# ── Step 3: Per-Hit Competition ──────────────────────────────
for angle in ANGLE_SETTINGS:
    fig, axes = plt.subplots(2, 4, figsize=(20, 8))
    axes_flat = axes.flatten()

    for i, n_trk in enumerate(TRACK_DENSITIES):
        key = f"a{angle}_n{n_trk}"
        ta_key = f"{key}_true_act"
        fs_key = f"{key}_false_sum"
        if ta_key not in step3_npz:
            continue
        true_act = step3_npz[ta_key]
        false_sum = step3_npz[fs_key]

        ax = axes_flat[i]
        if len(true_act) > 0:
            ax.scatter(false_sum, true_act, alpha=0.3, s=8, c='steelblue',
                       edgecolors='none')
            ax.axhline(THRESHOLD, color='green', ls='--', lw=1.2, alpha=0.7,
                       label=f'thr={THRESHOLD:.2f}')
            ax.plot([0, 1], [0, 1], 'k:', alpha=0.3)
        ax.set_title(f'{n_trk} tracks ({len(true_act):,} contested)', fontweight='bold')
        ax.set_xlabel('Sum of false competitor activations')
        ax.set_ylabel('True segment activation')
        ax.set_xlim(-0.05, 1.05)
        ax.set_ylim(-0.05, 1.05)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    fig.suptitle(f'Step 3: Per-Hit Competition — angle ±{angle} rad',
                 fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    save_fig(fig, f'hc_step3_competition_angle_{angle}')
    plt.show()

In [ ]:
# ── Step 4: Track-Level Reconstruction Quality ───────────────
import pandas as pd

df_reco = pd.DataFrame(reco_metrics)

for angle in ANGLE_SETTINGS:
    df_a = df_reco[df_reco['angle'] == angle]
    agg = df_a.groupby('n_tracks').agg(
        eff_mean=('efficiency', 'mean'),
        eff_std=('efficiency', 'std'),
        ghost_mean=('ghost_rate', 'mean'),
        ghost_std=('ghost_rate', 'std'),
        clone_mean=('clone_fraction', 'mean'),
        clone_std=('clone_fraction', 'std'),
    ).reset_index()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].errorbar(agg['n_tracks'], agg['eff_mean'] * 100,
                     yerr=agg['eff_std'] * 100,
                     fmt='o-', color='forestgreen', capsize=4,
                     markeredgecolor='black', markeredgewidth=0.5)
    axes[0].set_xlabel('Tracks per event')
    axes[0].set_ylabel('Efficiency (%)')
    axes[0].set_title('(a) Track Efficiency', fontweight='bold')
    axes[0].grid(True, alpha=0.3)

    axes[1].errorbar(agg['n_tracks'], agg['ghost_mean'] * 100,
                     yerr=agg['ghost_std'] * 100,
                     fmt='s-', color='indianred', capsize=4,
                     markeredgecolor='black', markeredgewidth=0.5)
    axes[1].set_xlabel('Tracks per event')
    axes[1].set_ylabel('Ghost Rate (%)')
    axes[1].set_title('(b) Ghost Rate', fontweight='bold')
    axes[1].grid(True, alpha=0.3)

    axes[2].errorbar(agg['n_tracks'], agg['clone_mean'] * 100,
                     yerr=agg['clone_std'] * 100,
                     fmt='D-', color='royalblue', capsize=4,
                     markeredgecolor='black', markeredgewidth=0.5)
    axes[2].set_xlabel('Tracks per event')
    axes[2].set_ylabel('Clone Fraction (%)')
    axes[2].set_title('(c) Clone Fraction', fontweight='bold')
    axes[2].grid(True, alpha=0.3)

    fig.suptitle(f'Step 4: Track-Level Reconstruction — angle ±{angle} rad',
                 fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    save_fig(fig, f'hc_step4_reco_angle_{angle}')
    plt.show()

    print(f"\nAngle ±{angle} rad:")
    print(agg.to_string(index=False))

In [ ]:
# ── Step 5: ROC / Threshold Sensitivity ──────────────────────
roc_raw = load_json(AGG_DIR / "hc_roc.json")

# Convert to usable format
roc_data = {}  # {angle: {n_trk: {'thresholds', 'eff', 'ghost'}}}
for a_str, trk_dict in roc_raw.items():
    angle = float(a_str)
    roc_data[angle] = {}
    for n_str, vals in trk_dict.items():
        n_trk = int(n_str)
        roc_data[angle][n_trk] = {
            'thresholds': np.array(vals['thresholds']),
            'eff': np.array(vals['eff']),
            'ghost': np.array(vals['ghost']),
        }

# ── ROC plot: one panel per angle ──
fig, axes = plt.subplots(1, len(ANGLE_SETTINGS), figsize=(6 * len(ANGLE_SETTINGS), 5))
if len(ANGLE_SETTINGS) == 1:
    axes = [axes]

dens_colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(ROC_DENSITIES)))

for a_idx, angle in enumerate(ANGLE_SETTINGS):
    ax = axes[a_idx]
    for d_idx, n_trk in enumerate(ROC_DENSITIES):
        if n_trk not in roc_data.get(angle, {}):
            continue
        rd = roc_data[angle][n_trk]
        ax.plot(rd['ghost'] * 100, rd['eff'] * 100,
                'o-', color=dens_colors[d_idx], lw=2, markersize=4,
                label=f'{n_trk} tracks')
        idx_def = np.argmin(np.abs(rd['thresholds'] - THRESHOLD))
        ax.plot(rd['ghost'][idx_def] * 100, rd['eff'][idx_def] * 100,
                '*', color=dens_colors[d_idx], markersize=14,
                markeredgecolor='black', markeredgewidth=0.8)
    ax.set_xlabel('Ghost Rate (%)')
    ax.set_ylabel('Efficiency (%)')
    ax.set_title(f'ROC — ±{angle} rad\n(★ = default thr)',
                 fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Step 5: Threshold Sensitivity / ROC — Angle Comparison',
             fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
save_fig(fig, 'hc_step5_roc_by_angle')
plt.show()

In [ ]:
# ── Step 5: Threshold curves (eff & ghost vs threshold) ──────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for angle in ANGLE_SETTINGS:
    for d_idx, n_trk in enumerate(ROC_DENSITIES):
        if n_trk not in roc_data.get(angle, {}):
            continue
        rd = roc_data[angle][n_trk]
        ls = '-' if angle == 0.2 else ('--' if angle == 0.1 else ':')
        axes[0].plot(rd['thresholds'], rd['eff'] * 100,
                     ls, color=dens_colors[d_idx], lw=1.5, alpha=0.7)
        axes[1].plot(rd['thresholds'], rd['ghost'] * 100,
                     ls, color=dens_colors[d_idx], lw=1.5, alpha=0.7)

axes[0].axvline(THRESHOLD, color='blue', ls='--', lw=1.2, alpha=0.6)
axes[0].set_xlabel('Threshold'); axes[0].set_ylabel('Efficiency (%)')
axes[0].set_title('(a) Efficiency vs Threshold', fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].axvline(THRESHOLD, color='blue', ls='--', lw=1.2, alpha=0.6)
axes[1].set_xlabel('Threshold'); axes[1].set_ylabel('Ghost Rate (%)')
axes[1].set_title('(b) Ghost Rate vs Threshold', fontweight='bold')
axes[1].grid(True, alpha=0.3)

legend_elements = [Line2D([0], [0], ls='-', color='black', label='±0.2 rad'),
                   Line2D([0], [0], ls='--', color='black', label='±0.1 rad'),
                   Line2D([0], [0], ls=':', color='black', label='±0.04 rad')]
axes[0].legend(handles=legend_elements, fontsize=9)
axes[1].legend(handles=legend_elements, fontsize=9)

plt.suptitle('Step 5: Threshold Curves — Angle Comparison',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, 'hc_step5_threshold_curves')
plt.show()

# ── Optimal threshold table ──
print(f"\n{'Angle':>7}  {'Tracks':>7}  {'Opt Thr':>8}  {'Eff (%)':>8}  {'Ghost (%)':>9}")
print('-' * 48)
for angle in ANGLE_SETTINGS:
    for n_trk in ROC_DENSITIES:
        if n_trk not in roc_data.get(angle, {}):
            continue
        rd = roc_data[angle][n_trk]
        mask = rd['ghost'] < 0.20
        if mask.any():
            best_idx = np.argmax(rd['eff'][mask])
            best_thr = rd['thresholds'][mask][best_idx]
            best_eff = rd['eff'][mask][best_idx]
            best_gr  = rd['ghost'][mask][best_idx]
        else:
            best_idx = np.argmax(rd['eff'])
            best_thr = rd['thresholds'][best_idx]
            best_eff = rd['eff'][best_idx]
            best_gr  = rd['ghost'][best_idx]
        print(f"  ±{angle:>4}  {n_trk:7d}  {best_thr:8.3f}  "
              f"{best_eff*100:7.1f}  {best_gr*100:8.1f}")
    print()

In [ ]:
# ── Step 6: Scattering comparison — segment metrics ──────────
hc_scatt_raw = load_json(AGG_DIR / "hc_scatt.json")
hc_scan_data = hc_scatt_raw["scan_data"]   # {mult_str: results_list}
hc_epsilons = hc_scatt_raw["epsilons"]

fig, axes = plt.subplots(2, 2, figsize=(15, 11))

for mult in SCATT_MULTIPLIERS:
    results = hc_scan_data[str(mult)]
    eps = float(hc_epsilons[str(mult)])
    col = scatter_colors[mult]
    fmt = scatter_fmts[mult]
    lbl = f'{mult}x ($\\varepsilon$={eps*1e3:.2f} mrad)'

    x_tracks = np.array([r['n_tracks'] for r in results])

    # (a) Segment Efficiency
    ax = axes[0, 0]
    eff_mean = np.array([r['eff_mean'] for r in results])
    eff_se   = np.array([r['eff_se']   for r in results])
    ax.errorbar(x_tracks, eff_mean, yerr=eff_se, fmt=fmt, color=col,
                capsize=4, capthick=1.2, markeredgecolor='black',
                markeredgewidth=0.6, label=lbl)

    # (b) False Rate
    ax = axes[0, 1]
    fr_mean = np.array([r['fr_mean'] for r in results])
    fr_se   = np.array([r['fr_se']   for r in results])
    ax.errorbar(x_tracks, fr_mean, yerr=fr_se, fmt=fmt, color=col,
                capsize=4, capthick=1.2, markeredgecolor='black',
                markeredgewidth=0.6, label=lbl)

    # (c) Segment Pair Counts
    ax = axes[1, 0]
    n_true_m  = np.array([r['n_true_mean']  for r in results])
    n_true_e  = np.array([r['n_true_se']    for r in results])
    n_false_m = np.array([r['n_false_mean'] for r in results])
    n_false_e = np.array([r['n_false_se']   for r in results])
    ax.errorbar(x_tracks, n_true_m,  yerr=n_true_e,  fmt=fmt, color=col,
                capsize=3, capthick=1, markeredgecolor='black',
                markeredgewidth=0.5, label=f'{mult}x true')
    ax.errorbar(x_tracks, n_false_m, yerr=n_false_e, fmt=fmt, color=col,
                capsize=3, capthick=1, markeredgecolor='black',
                markeredgewidth=0.5, alpha=0.4, label=f'{mult}x false')

    # (d) Accepted
    ax = axes[1, 1]
    ta_m = np.array([r['true_acc_mean']  for r in results])
    ta_e = np.array([r['true_acc_se']    for r in results])
    fa_m = np.array([r['false_acc_mean'] for r in results])
    fa_e = np.array([r['false_acc_se']   for r in results])
    fa_m_plot = np.where(fa_m > 0, fa_m, 0.5)
    fa_e_plot = np.where(fa_m > 0, fa_e, 0)
    ax.errorbar(x_tracks, ta_m, yerr=ta_e, fmt=fmt, color=col,
                capsize=3, capthick=1, markeredgecolor='black',
                markeredgewidth=0.5, label=f'{mult}x true acc')
    ax.errorbar(x_tracks, fa_m_plot, yerr=fa_e_plot, fmt=fmt, color=col,
                capsize=3, capthick=1, markeredgecolor='black',
                markeredgewidth=0.5, alpha=0.4, label=f'{mult}x false acc')

# Axis formatting
axes[0, 0].axhline(100, color='gray', ls='--', lw=1, alpha=0.5)
axes[0, 0].set_xlabel('Tracks per event')
axes[0, 0].set_ylabel('Segment Efficiency (%)')
axes[0, 0].set_title('(a) Segment Efficiency\n'
                      r'($N_{\mathrm{true\,accepted}} / N_{\mathrm{true\,pairs}}$)',
                      fontweight='bold')
axes[0, 0].legend(fontsize=10, loc='lower left')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].set_xlabel('Tracks per event')
axes[0, 1].set_ylabel('Segment False Rate (%)')
axes[0, 1].set_title('(b) Segment False Rate\n'
                      r'($N_{\mathrm{false\,accepted}} / N_{\mathrm{all\,accepted}}$)',
                      fontweight='bold')
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

for ax_idx in [(1, 0), (1, 1)]:
    axes[ax_idx].set_yscale('log')
    axes[ax_idx].set_xlabel('Tracks per event')
    axes[ax_idx].grid(True, alpha=0.3, which='both')

axes[1, 0].set_ylabel('Number of segment pairs')
axes[1, 0].set_title('(c) Segment Pair Counts', fontweight='bold')
axes[1, 0].legend(fontsize=9, ncol=2, loc='upper left')

axes[1, 1].set_ylabel('Number of pairs accepted')
axes[1, 1].set_title(r'(d) Accepted Segment Pairs ($\theta \leq \varepsilon$)',
                      fontweight='bold')
axes[1, 1].legend(fontsize=9, ncol=2, loc='upper left')

fig.suptitle('Step 6: Segment-Level Metrics vs Event Size — Scattering Comparison\n'
             f'({N_REPEATS_SCAN} repeats per size)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, 'hc_step6_scattering_comparison')
plt.show()

# Summary tables
for mult in SCATT_MULTIPLIERS:
    eps = float(hc_epsilons[str(mult)])
    results = hc_scan_data[str(mult)]
    print(f"\n{'='*95}")
    print(f"  {mult}x scattering  (sigma_scatt={SIGMA_SCATT*mult:.1e}, "
          f"epsilon={eps*1e3:.3f} mrad)")
    print(f"{'='*95}")
    print(f"{'Tracks':>7} {'Eff (%)':>12} {'FR (%)':>12} "
          f"{'True pairs':>14} {'False pairs':>14} "
          f"{'True acc':>14} {'False acc':>14}")
    print('-' * 95)
    for r in results:
        print(f"{r['n_tracks']:7d} "
              f"{r['eff_mean']:7.1f}+/-{r['eff_se']:.2f} "
              f"{r['fr_mean']:7.2f}+/-{r['fr_se']:.2f} "
              f"{r['n_true_mean']:8.0f}+/-{r['n_true_se']:.0f} "
              f"{r['n_false_mean']:8.0f}+/-{r['n_false_se']:.0f} "
              f"{r['true_acc_mean']:8.0f}+/-{r['true_acc_se']:.0f} "
              f"{r['false_acc_mean']:8.0f}+/-{r['false_acc_se']:.0f}")

In [ ]:
# ── Step 6: Scattering acceptance histograms ─────────────────
hc_hist_npz = np.load(str(AGG_DIR / "hc_scatt_hist.npz"))

x_max_rad = 0.03
bins = np.linspace(0, x_max_rad, 100)

fig, axes = plt.subplots(len(SCATT_MULTIPLIERS), 3,
                         figsize=(20, 5 * len(SCATT_MULTIPLIERS)))

for row, mult in enumerate(SCATT_MULTIPLIERS):
    eps = all_epsilons[mult]
    ta = hc_hist_npz[f"m{mult}_true"]
    fa = hc_hist_npz[f"m{mult}_false"]

    ax = axes[row, 0]
    ax.hist(ta, bins=bins, color='forestgreen', alpha=0.8,
            edgecolor='black', lw=0.3, label=f'True ({len(ta):,})')
    ax.axvline(eps, color='blue', ls='--', lw=2,
               label=f'$\\varepsilon$ = {eps*1e3:.2f} mrad')
    ax.set_title(f'{mult}x scattering - True Segment Pairs',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Pairwise angle (rad)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    ax = axes[row, 1]
    ax.hist(fa, bins=bins, color='indianred', alpha=0.8,
            edgecolor='black', lw=0.3, label=f'False ({len(fa):,})')
    ax.axvline(eps, color='blue', ls='--', lw=2,
               label=f'$\\varepsilon$ = {eps*1e3:.2f} mrad')
    ax.set_title(f'{mult}x scattering - False Segment Pairs',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Pairwise angle (rad)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    ax = axes[row, 2]
    ax.hist(ta, bins=bins, color='forestgreen', alpha=0.6,
            edgecolor='black', lw=0.3, density=True,
            label=f'True ({len(ta):,})')
    ax.hist(fa, bins=bins, color='indianred', alpha=0.5,
            edgecolor='black', lw=0.3, density=True,
            label=f'False ({len(fa):,})')
    ax.axvline(eps, color='blue', ls='--', lw=2,
               label=f'$\\varepsilon$ = {eps*1e3:.2f} mrad')
    ax.set_title(f'{mult}x scattering - Normalised overlay',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Pairwise angle (rad)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

N_TRACKS_HIST = params.get('n_tracks_hist', 50)
N_EVENTS_HIST = params.get('n_events_hist', 20)
fig.suptitle(f'Step 6: Segment-Pair Angle Distributions — {N_EVENTS_HIST} events × '
             f'{N_TRACKS_HIST} tracks per scattering setting',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig(fig, 'hc_step6_acceptance_histograms')
plt.show()

---
# Summary

All plots have been reproduced from the aggregated Condor pipeline results.
The data is saved in `results/aggregated/` and figures in `results/figures/`.

To re-run the analysis:
1. Simply re-run this notebook — it loads from saved data.
2. To regenerate from scratch: `./submit.sh` → wait → `python scripts/aggregate.py --results-dir results` → re-run notebook.